# Exercise 2.2.2.7 — implement `VPGTrainer`

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `2.2.2 Policy Gradient`  
**Notebook:** `2.2.2_Policy_Gradient_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=2.2.2.7](https://delta-drills.vercel.app/?arena_exercise=2.2.2.7)


# [2.2.2] - Vanilla Policy Gradient (VPG) (exercises)

> **ARENA [Streamlit Page](https://arena-chapter2-rl.streamlit.app/21_[2.2.2]_Policy_Gradient)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part22_vpg/2.2.2_Policy_Gradient_exercises.ipynb?t=20250917) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part22_vpg/2.2.2_Policy_Gradient_solutions.ipynb?t=20250917)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

# Introduction

You'll also implement Vanilla Policy Gradient (VPG), the first policy gradient algorithm upon which many modern RL algorithms are based (including PPO).

## Content & Learning Objectives

### 1️⃣ Policy Gradient Theorem

The Policy Gradient Theorem is what all policy gradient methods are based on: it allows us to compute the gradient of the return, something that would naively not have a well defined gradient.

> ##### Learning Objectives
>
> - Understand the Policy Gradient Theorem

### 2️⃣ Implementation


> ##### Learning Objectives
>
> - Understand the VPG algorithm: how to perform on-policy policy gradient
> - Implement VPG using PyTorch, on the CartPole environment

## 🚧 Under construction 🚧

This material is still in beta, and may be severely lacking in tests, or have bugs. Please report problems you find in `#errata`!

## Optional Readings

* [Policy Gradient Algorithms](https://lilianweng.github.io/posts/2018-04-08-policy-gradient/) (25 minutes)
    * Skip the derivation of the policy gradient theorem, we've already done that here.
    * Covers many other policy gradient methods we don't, you may wish to implement some of them afterwards as a bonus.

## Setup code

In [ ]:
from __future__ import annotations

import os
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, TypeAlias, Optional

import gymnasium as gym
import numpy as np
import torch as t
import wandb
from gymnasium.spaces import Box, Discrete
from jaxtyping import Bool, Float, Int
from torch import Tensor, nn
from tqdm import tqdm, trange
import torch.nn.functional as F
from torch.utils.data import DataLoader

from eindex import eindex

warnings.filterwarnings("ignore")

ActType = Int
ObsType = Int

In [ ]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# Make sure exercises are in the path
chapter = "chapter2_rl"
section = "part21_dqn"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part22_vpg.tests as tests
import part22_vpg.utils as utils
from part1_intro_to_rl.solutions import Environment, Norvig, Toy, find_optimal_policy
from part1_intro_to_rl.utils import set_global_seeds
from rl_utils import make_env
from plotly_utils import cliffwalk_imshow, line, plot_cartpole_obs_and_dones
from rl_utils import generate_and_plot_trajectory


from gpu_env import CartPole
from probe import Probe4, Probe5
from collections import namedtuple
from torch.utils.data import Dataset, TensorDataset

from torchinfo import summary


device = t.device(
   "mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu"
)

MAIN = __name__ == "__main__"

# 1️⃣ Policy Gradient Theorem

> ##### Learning Objectives
>
> - Understand the Policy Gradient Theorem

Instead of learning action-values and deriving a policy (as in Q-learning or DQN), **policy gradient methods learn the policy directly**.  
- Policy is parameterized: $\pi_\theta(a|s)$ with parameters $\theta$ (often a neural network).  
- Objective: Choose $\theta$ to maximize expected return $J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[G(\tau)]$ (joy), where $\tau$ is a trajectory and $G(\tau)$ its return.  

We would desire to update the policy directly via **gradient ascent** against $J(\theta)$:
$$
\theta \leftarrow \theta + \alpha \nabla_\theta J(\theta)
$$

The problem is that the return is a sum of rewards from the trajectory, and the trajectory itself is a result of sampling from the policy, over and over, 
as well as being dependant on the environmental distribution, which we do not have access to.
There is no clear way to directly compute the gradient of the return with respect to the policy parameters.
The solution here is the **policy gradient theorem**, which states that we can instead use the log-probability weighted return as an unbiased estimator of the gradient of the return.

$$
\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_t G_t \nabla_\theta \log \pi_\theta(a_t|s_t) \right]
$$

<details>
<summary>Derivation</summary>

The probability of sampling a trajectory 
$\tau = (s_0, a_0, s_1, a_1, \dots, s_T)$ 
is given by
$$
\Pr(\tau|\theta) = \prod_{t=0}^{T-1} \pi_\theta(a_t|s_t)\,\mu(s_{t+1}|s_t, a_t)
$$
where $\mu$ is the environment transition probability.

$$
\begin{align*}
   \nabla_\theta J(\theta) &= \nabla_\theta \mathbb{E}_{\tau \sim \pi_\theta}[G(\tau)] \\
   &= \nabla_\theta \sum_\tau  \Pr(\tau|\theta) \, G(\tau) \\
   &= \sum_\tau \nabla_\theta \Pr(\tau|\theta) \, G(\tau) \\
   &= \sum_\tau \Pr(\tau|\theta)\,\nabla_\theta \log \Pr(\tau|\theta) \, G(\tau) \\
   &= \mathbb{E}_{\tau \sim \pi_\theta}\left[ \nabla_\theta \log \Pr(\tau|\theta) \, G(\tau) \right]
   \end{align*}
   $$
   where we made use of the log-derivative trick: $\nabla_\theta p(x) = p(x) \nabla_\theta \log p(x)$.
  
 
   The dynamics $\mu$ do not depend on $\theta$, so:
   $$
   \begin{align*}
   \log \Pr(\tau|\theta) &= \log \left( \prod_{t=0}^{T-1} \pi_\theta(a_t|s_t)\,\mu(s_{t+1}|s_t, a_t) \right) \\
   &= \sum_{t=0}^{T-1} \log \pi_\theta(a_t|s_t) + \sum_{t=0}^{T-1} \log \mu(s_{t+1}|s_t, a_t) \\
   &= \sum_{t=0}^{T-1} \log \pi_\theta(a_t|s_t) + \text{const.}
   \end{align*}
   $$
   where the const. term is independent of $\theta$, so when we take the gradient, it vanishes.

   Thus:
   $$
   \nabla_\theta \log \Pr(\tau|\theta) = \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t)
   $$

Plugging back into the gradient:
$$
\nabla_\theta J(\theta) =
\mathbb{E}_{\tau \sim \pi_\theta} \left[
  \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t)\, G(\tau)
\right]
$$

This is the **Vanilla Policy Gradient estimator**, also called **REINFORCE**. 
Each $\log \pi_\theta(a_t|s_t)$ is multiplied by the **full return** $G(\tau)$. 
However, the action $a_t$ cannot influence rewards before time $t$, only those afterwards.
This means that all the rewards before timestep $t$ merely add noise, as no changes to the policy
can affect them.  To reduce variance, replace $G(\tau)$ with the return $G_t$ at timestep $t$, 
also called the **reward-to-go**:
$$
G_t = \sum_{i=t}^{T} \gamma^{i-t} r_{i}
$$

Thus, the lower-variance unbiased estimator is:
$$
\nabla_\theta J(\theta) =
\mathbb{E}_{\tau \sim \pi_\theta} \left[
  \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t)\, G_t
\right]
$$

</details>

There are many other variants of the policy gradient estimator, as described in [Schulman, 2018](https://arxiv.org/abs/1506.02438).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/img/policy_grad.png" width="800">

# 2️⃣ Implementation

> ##### Learning Objectives
>
> - Understand the VPG algorithm: how to perform on-policy policy gradient
> - Implement VPG using PyTorch, on the CartPole environment

We make use of the same CartPole environment as before, but now we have a vectorized version that is entirely defined in terms of tensor operations (see `chapter2_rl/exercises/gpu_env.py`). This environment is identical to the one used for DQN, but it now runs entirely on the GPU. This means
* we don't need to constantly convert between numpy and torch tensors
* we can run large numbers of environments in parallel (~thousands of environments for ~millions of environmental steps per second)
* we avoid copying data back and forth between the CPU and GPU, which can be a significant bottleneck

## Policy Network

Here, the policy is learned directly as a neural network, rather than learning a Q-value table approximator. We'll use the same architecture as the Q-network from DQN, so we've just included that here for you.

In [ ]:
class PolicyNetwork(nn.Module):
    """
    For consistency with your tests, please wrap your modules in a `nn.Sequential` called `layers`.
    """

    layers: nn.Sequential


    def __init__(
        self, obs_shape: tuple[int], num_actions: int, hidden_sizes: list[int] = [120, 84]
    ):
        super().__init__()
        #assert len(obs_shape) == 1, f"Expecting a single vector of observations, got {obs_shape}"
        assert len(hidden_sizes) == 2, f"Expecting 2 hidden layers, got {len(hidden_sizes)}"
        self.layers = nn.Sequential(nn.Linear(obs_shape[-1], hidden_sizes[0]),
                                    nn.ReLU(),
                                    nn.Linear(hidden_sizes[0], hidden_sizes[1]),
                                    nn.ReLU(),
                                    nn.Linear(hidden_sizes[1], num_actions))

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)

net = PolicyNetwork(obs_shape=(4,), num_actions=2)
summary(net)

## Rollout Buffer

The way that our implementation of VPG will work is simple: we perform a rollout acrosss `num_envs` many environments in parallel, and store the trajectories for each. We then learn from that set of rollouts, and then discard it afterwards. One rollout, one learning step. This means we are always learning **on-policy**: we only every learn from data that the current model actually generated. We will use a rollout buffer to store the trajectories.

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "2.2.2.7"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part22_vpg.solutions import Rollout, VPGAgent, compute_returns, compute_logprobs_and_entropy, compute_importance_weights, normalize_returns, compute_reinforce_loss


### Exercise - implement `VPGTrainer`

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 45 minutes on this exercise.
> ```

You should fill in the following methods. Ignore logging, can just copy from the solution later.

* `compute_loss` - this method should compute the loss for the VPG objective function.

The training loop is rather standard once everything else is done: we do a rollout, we cut the result into batches, compute the loss, and update the weights from each batch, so we've just included that gor

In [ ]:
class VPGTrainer:
    def __init__(self, args: VPGArgs):
        set_global_seeds(args.seed)
        self.args = args
        
        device = args.device
        
        self.rng = t.Generator(device=device).manual_seed(args.seed)
        self.run_name = f"{args.env_id}__{args.wandb_project_name}__seed{args.seed}__{time.strftime('%Y%m%d-%H%M%S')}"
        
        if args.env_id=="CartPole-gpu":
            self.envs = CartPole(args.num_envs, device = device)
        elif args.env_id == "Probe4-v0":
            self.envs = Probe4(args.num_envs)
        elif args.env_id == "Probe5-v0":
            self.envs = Probe5(args.num_envs)
        else:
            raise ValueError(f"Environment {args.env_id} not supported")

        # Define some basic variables from our environment (note, we assume a single discrete action space)
        self.num_envs = args.num_envs
        self.action_shape = self.envs.action_space.shape
        self.num_actions = self.envs.action_space.n
        self.obs_shape = self.envs.observation_space.shape

        # Create our networks & optimizer
        self.policy_network = PolicyNetwork(self.obs_shape, self.num_actions).to(device)
        
        # Compile the policy network for faster inference
        if self.args.compile:
            self.policy_network = t.compile(self.policy_network)
        
        self.optimizer = t.optim.Adam(self.policy_network.parameters(), 
                                       lr=args.lr,
                                       eps=1e-5,
                                       maximize=True)
        self.optimizer.zero_grad()

        # Create our agent
        self.agent = VPGAgent(
            envs=self.envs, 
            policy_network=self.policy_network, 
            args=self.args,
            rng=self.rng
        )

    def compute_loss(self, tau: RolloutTensors
    ) -> tuple[t.Tensor, dict[str, Any]]:
        
        raise NotImplementedError()

        info = {
            "entropy": avg_entropy.item(),
            "r_joy": r_joy.item(),
            "iw": iw.mean().item() if self.args.use_iw else None,
        }

        return joy, info
       

    def update_learning_rate(self, time_steps, args):
        if args.use_lr_decay and args.lr_frac > 0:
            progress = min(1.0, max(time_steps / args.total_timesteps, 0) / args.lr_frac)
            return (progress * args.lr_end) + ((1 - progress) * args.lr)
        return args.lr


    def train(self) -> None:
        """
        Trains the agent by generating rollouts and updating the policy.
        The progress bar tracks total environment steps.
        """
        # --- Setup ---
        rollout = Rollout(num_envs=self.num_envs, 
                          max_steps=self.args.num_steps_per_rollout, 
                          obs_shape=self.obs_shape, 
                          action_shape=self.action_shape, 
                          device=self.args.device)
        
        # Calculate the number of environment steps collected per rollout generation
        
        
        # Calculate the total number of updates (rollouts) to perform
        # Use integer division to ensure we don't exceed total_timesteps
        
        env_steps_per_train_step = (self.args.num_steps_per_rollout * self.args.num_envs
                                    // (self.args.num_batches_per_rollout))
        
        num_updates = self.args.total_timesteps // env_steps_per_train_step
        train_steps = 0  # Counter for gradient updates
        
        # --- Training Loop ---
        # The progress bar is managed manually with a `with` statement.
        # `total` is set to the total environment steps we want to run.
        # The loop iterates `num_updates` times, not `total_timesteps` times.
        with tqdm(total=self.args.total_timesteps, unit=" env steps", unit_scale=True, 
                  desc="Training", miniters=1, mininterval=0.02) as pbar:
            
            env_steps_consumed = 0
            
            for update_num in range(num_updates):
                # 1. Generate a new rollout from the environment
                
                rollout, agent_info = self.agent.gen_rollout(rollout)

                # 2. Split the rollout into batches along the num_envs dimension
              
                rollout_batches = rollout.get_batches(self.args.batch_size)
                
                # 3. Logging and Progress Bar Update
                # This part is outside the inner loop to only log once per rollout
                avg_lifespan = agent_info["lifespan"].float().mean().item()
                std_lifespan = agent_info["lifespan"].float().std().item()
                max_lifespan = agent_info["lifespan"].max().item()
                
                if (avg_lifespan + 0.5) > self.args.num_steps_per_rollout \
                    and std_lifespan < 0.01:
                    print("Agent has learned to play optimally!")
                    break
                
                # 4. For each batch, perform multiple gradient updates
                for batch in rollout_batches:
                    for i in range(self.args.rollout_use_count):
                        loss, reinforce_info = self.compute_loss(batch)
                        
                        info = {**agent_info, **reinforce_info}
                        
                        loss.backward()
                        if self.args.max_grad_norm is not None:
                            t.nn.utils.clip_grad_norm_(self.policy_network.parameters(), max_norm=self.args.max_grad_norm)
                            
                        grad_norm = t.nn.utils.clip_grad_norm_(self.policy_network.parameters(), max_norm=float('inf'))
                        
                        self.optimizer.step()
                        self.optimizer.zero_grad()
                        train_steps += 1
          
                        new_lr = self.update_learning_rate(pbar.n, self.args)
                        
                        for pg in self.optimizer.param_groups:
                            pg["lr"] = new_lr
                    
                        # Create info string to display in the progress bar
                        current_lr = self.optimizer.param_groups[0]["lr"]
                        info_dict = {
                            "joy": f"{info['r_joy']:.4f}",
                            "traj_len": f"{avg_lifespan:.2f} ± {std_lifespan:.2f} (max: {max_lifespan:.2f})",
                            "H": f"{info['entropy']:.4f}",
                            "iw": f"{info['iw']:.4f}" if self.args.use_iw else None,
                            "∇": f"{grad_norm:.4f}",
                            "lr": f"{current_lr:.2e}",
                        }

                        pbar.set_postfix(info_dict)
                        pbar.update(env_steps_per_train_step)
            
        # --- Cleanup ---
        self.envs.close()
        if self.args.use_wandb:
            wandb.finish()

<details><summary>Solution</summary>

```python
class VPGTrainer:
    def __init__(self, args: VPGArgs):
        set_global_seeds(args.seed)
        self.args = args
        
        device = args.device
        
        self.rng = t.Generator(device=device).manual_seed(args.seed)
        self.run_name = f"{args.env_id}__{args.wandb_project_name}__seed{args.seed}__{time.strftime('%Y%m%d-%H%M%S')}"
        
        if args.env_id=="CartPole-gpu":
            self.envs = CartPole(args.num_envs, device = device)
        elif args.env_id == "Probe4-v0":
            self.envs = Probe4(args.num_envs)
        elif args.env_id == "Probe5-v0":
            self.envs = Probe5(args.num_envs)
        else:
            raise ValueError(f"Environment {args.env_id} not supported")

        # Define some basic variables from our environment (note, we assume a single discrete action space)
        self.num_envs = args.num_envs
        self.action_shape = self.envs.action_space.shape
        self.num_actions = self.envs.action_space.n
        self.obs_shape = self.envs.observation_space.shape

        # Create our networks & optimizer
        self.policy_network = PolicyNetwork(self.obs_shape, self.num_actions).to(device)
        
        # Compile the policy network for faster inference
        if self.args.compile:
            self.policy_network = t.compile(self.policy_network)
        
        self.optimizer = t.optim.Adam(self.policy_network.parameters(), 
                                       lr=args.lr,
                                       eps=1e-5,
                                       maximize=True)
        self.optimizer.zero_grad()

        # Create our agent
        self.agent = VPGAgent(
            envs=self.envs, 
            policy_network=self.policy_network, 
            args=self.args,
            rng=self.rng
        )

    def compute_loss(self, tau: RolloutTensors
    ) -> tuple[t.Tensor, dict[str, Any]]:
        
        returns = compute_returns(tau.rewards, tau.dones, self.args.gamma)  # (num_envs, timestep)

        if self.args.normalize_returns:
            returns = normalize_returns(returns)

        logprobs_taken, entropy = compute_logprobs_and_entropy(tau, self.policy_network)

        iw = compute_importance_weights(logprobs_taken, tau, self.args.clip_coef)
        r_joy = compute_reinforce_loss(returns, logprobs_taken, iw)
        avg_entropy = entropy.mean()

        joy = r_joy + self.args.ent_coef * avg_entropy
        

        info = {
            "entropy": avg_entropy.item(),
            "r_joy": r_joy.item(),
            "iw": iw.mean().item() if self.args.use_iw else None,
        }

        return joy, info
       

    def update_learning_rate(self, time_steps, args):
        if args.use_lr_decay and args.lr_frac > 0:
            progress = min(1.0, max(time_steps / args.total_timesteps, 0) / args.lr_frac)
            return (progress * args.lr_end) + ((1 - progress) * args.lr)
        return args.lr


    def train(self) -> None:
        """
        Trains the agent by generating rollouts and updating the policy.
        The progress bar tracks total environment steps.
        """
        # --- Setup ---
        rollout = Rollout(num_envs=self.num_envs, 
                          max_steps=self.args.num_steps_per_rollout, 
                          obs_shape=self.obs_shape, 
                          action_shape=self.action_shape, 
                          device=self.args.device)
        
        # Calculate the number of environment steps collected per rollout generation
        
        
        # Calculate the total number of updates (rollouts) to perform
        # Use integer division to ensure we don't exceed total_timesteps
        
        env_steps_per_train_step = (self.args.num_steps_per_rollout * self.args.num_envs
                                    // (self.args.num_batches_per_rollout))
        
        num_updates = self.args.total_timesteps // env_steps_per_train_step
        train_steps = 0  # Counter for gradient updates
        
        # --- Training Loop ---
        # The progress bar is managed manually with a `with` statement.
        # `total` is set to the total environment steps we want to run.
        # The loop iterates `num_updates` times, not `total_timesteps` times.
        with tqdm(total=self.args.total_timesteps, unit=" env steps", unit_scale=True, 
                  desc="Training", miniters=1, mininterval=0.02) as pbar:
            
            env_steps_consumed = 0
            
            for update_num in range(num_updates):
                # 1. Generate a new rollout from the environment
                
                rollout, agent_info = self.agent.gen_rollout(rollout)

                # 2. Split the rollout into batches along the num_envs dimension
              
                rollout_batches = rollout.get_batches(self.args.batch_size)
                
                # 3. Logging and Progress Bar Update
                # This part is outside the inner loop to only log once per rollout
                avg_lifespan = agent_info["lifespan"].float().mean().item()
                std_lifespan = agent_info["lifespan"].float().std().item()
                max_lifespan = agent_info["lifespan"].max().item()
                
                if (avg_lifespan + 0.5) > self.args.num_steps_per_rollout \
                    and std_lifespan < 0.01:
                    print("Agent has learned to play optimally!")
                    break
                
                # 4. For each batch, perform multiple gradient updates
                for batch in rollout_batches:
                    for i in range(self.args.rollout_use_count):
                        loss, reinforce_info = self.compute_loss(batch)
                        
                        info = {**agent_info, **reinforce_info}
                        
                        loss.backward()
                        if self.args.max_grad_norm is not None:
                            t.nn.utils.clip_grad_norm_(self.policy_network.parameters(), max_norm=self.args.max_grad_norm)
                            
                        grad_norm = t.nn.utils.clip_grad_norm_(self.policy_network.parameters(), max_norm=float('inf'))
                        
                        self.optimizer.step()
                        self.optimizer.zero_grad()
                        train_steps += 1
          
                        new_lr = self.update_learning_rate(pbar.n, self.args)
                        
                        for pg in self.optimizer.param_groups:
                            pg["lr"] = new_lr
                    
                        # Create info string to display in the progress bar
                        current_lr = self.optimizer.param_groups[0]["lr"]
                        info_dict = {
                            "joy": f"{info['r_joy']:.4f}",
                            "traj_len": f"{avg_lifespan:.2f} ± {std_lifespan:.2f} (max: {max_lifespan:.2f})",
                            "H": f"{info['entropy']:.4f}",
                            "iw": f"{info['iw']:.4f}" if self.args.use_iw else None,
                            "∇": f"{grad_norm:.4f}",
                            "lr": f"{current_lr:.2e}",
                        }

                        pbar.set_postfix(info_dict)
                        pbar.update(env_steps_per_train_step)
            
        # --- Cleanup ---
        self.envs.close()
        if self.args.use_wandb:
            wandb.finish()
```
</details>

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
